# Interop with Microsoft resource estimator

Having interoperability with Microsoft's open-source resource estimator is useful because it allows users more flexibility with their FlagQuantum programs and allows users to verify estimates given by FlagQuantum's resource estimator tooling.

In this notebook, we will demonstrate doing resource estimation on the `ModularExponentiation` component, the essential subroutine in Shor's factoring algorithm.

In [ ]:
from importlib import import_module

from flagquantum.algorithms.primitives.modular_exponentiation import (
    ModularExponentiation,
)
from flagquantum.drawer.signature_graph import export_signature_graph_dot

N = 13 * 17
g = 8
mod_exp = ModularExponentiation(base=g, modulus=N, exponent_width=32, output_width=32)
component_graph = export_signature_graph_dot(
    mod_exp.source_signature, mod_exp.source_label
)
import_module("graphviz").Source(component_graph)


In the next cell we have a function which takes in a native component and returns LogicalCounts, a python object which stores information about the logical data and gate counts. You can initialize the LogicalCounts object with a dictionary with information about your quantum algorithm. You can populate fields such as `numQubits` and `tCount` with information from a native declared workload as show below. For a more complete overview of the fields which you can specify in LogicalCounts, click [here](https://learn.microsoft.com/en-us/azure/quantum/resource-estimator-known-estimates?tabs=tabid-known-estimates-qsharp) and look for the table which specifies the available functions with AccountForEstimates.

In [ ]:
from flagquantum.compiler._component_trace.resource_model import (
    GateCost,
    ModeledResources,
)
from flagquantum.ecosystem.resource_estimator import (
    MicrosoftLogicalCounts,
    microsoft_logical_counts,
)


def get_modular_exponentiation_counts(
    operation: ModularExponentiation,
) -> MicrosoftLogicalCounts:
    # The source declares e controlled multipliers, each with two scaled adds
    # and n CSwaps. Each scaled add declares n*(5*n+2) forward ANDs.
    e, n = operation.exponent_width, operation.output_width
    if type(e) is not int or type(n) is not int:
        raise ValueError("This tutorial requires concrete register widths")
    gates = GateCost(and_bloq=e * 2 * n * (5 * n + 2), cswap=e * n)
    # Match the original signature-only export. This omits temporary work.
    signature_width = operation.source_signature.n_qubits()
    if type(signature_width) is not int:
        raise ValueError("This tutorial requires concrete signature width")
    return microsoft_logical_counts(ModeledResources(gates, signature_width))


The LogicalCounts object has the key method, `estimate`, which returns an `EstimatorResult` which contains the physical resource estimates. By running the cell below, you can see a nice display of the physical resource estimates including runtime, physical qubits, and many more important resource estimates. For a more complete description of the `EstimatorResult`, click [here](https://learn.microsoft.com/en-us/azure/quantum/overview-resource-estimator-output-data).

In [ ]:
LogicalCounts = import_module("qsharp.estimator").LogicalCounts
estimates = LogicalCounts(get_modular_exponentiation_counts(mod_exp)).estimate(
    [
        {"qubitParams": {"name": "qubit_gate_ns_e3"}},
        {"qubitParams": {"name": "qubit_gate_ns_e4"}},
        {
            "qubitParams": {"name": "qubit_maj_ns_e4"},
            "surfaceCode": {"name": "floquet_code"},
        },
        {
            "qubitParams": {"name": "qubit_maj_ns_e6"},
            "surfaceCode": {"name": "floquet_code"},
        },
    ]
)
estimates


You can display more complex UIs on top of estimates with qsharp_widgets. For example, the code below displays a chart with information on the physical qubit counts. To learn more about using qsharp_widgets to create other informative graphics such as a Space-time diagram, click [here](https://learn.microsoft.com/en-us/azure/quantum/overview-resource-estimator-output-data#space-time-diagram).

In [ ]:
SpaceChart = import_module("qsharp_widgets").SpaceChart
SpaceChart(estimates)


## Source, scope and optional dependencies
Adapted from all eight cells of Qualtran v0.7.0, revision `bcb17bb25360ed68ce3bc084645db3e929799389`, blob `25b2ba4cb6b9073444016e3c89e72f9de4a0c2e8`. Copyright 2023 Google LLC, Apache-2.0.
The local reference uses qsharp 1.9.0 and qsharp-widgets 1.9.0. These optional packages perform local analytical estimation and widget display. The native count converter imports neither. SDK objects stay in this interoperability example.
The upstream export uses 64 signature qubits, while the full component peak is 195. Its T count is 1,334,272. This notebook reproduces that signature-only estimate and its undercount explicitly. Production callers can supply complete peak logical resources to the same converter. The four hardware configurations are external estimator results, separate from emitted primitive counts and native physical models. This example does not execute a 32-bit factoring circuit. Circuit construction retains `circuit.append(op)`.
